# 📱 Análisis Exploratorio del Modelo Comercial de Celulares

---

| Campo | Detalle |
|---|---|
| **Módulo** | Friendly SQL & Python para Analytics |
| **Diploma** | Data Analyst — DMC Institute |
| **Dataset** | ModeloComercial.xlsx (fuente propia) |
| **Tecnologías** | Python · pandas · numpy · duckdb · plotly |

---

## 🎯 Objetivo del trabajo

Desarrollar un análisis exploratorio completo sobre el dataset de ventas de una empresa distribuidora de smartphones. El análisis integra carga de datos, exploración, estadística descriptiva, visualizaciones interactivas con Plotly y consultas SQL mediante DuckDB, con el fin de identificar patrones de rentabilidad, desempeño por tienda y comportamiento de clientes.

## 📦 Descripción del dataset

El dataset contiene **1,499 facturas de venta** registradas entre **2014 y 2020**, distribuidas en 8 hojas relacionadas:
- **Ventas** — tabla principal con transacciones
- **Tienda, Vendedor, Cliente, País, Modelo, Marca, Operador** — tablas de dimensión

**Variables clave:** Fecha, Cantidad, Costo, Precio, Venta, FechaVencimiento, FechaPago

---
## ⚙️ 0. Instalación de librerías
Instalamos las dependencias necesarias si no están disponibles en el entorno.

In [ ]:
# Instalar librerías si es necesario (descomentar en Google Colab)
# !pip install duckdb plotly openpyxl -q

import pandas as pd
import numpy as np
import duckdb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías cargadas correctamente')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
print(f'   duckdb  {duckdb.__version__}')

---
## 📂 1. Carga del Dataset

Leemos el archivo Excel con todas sus hojas usando `pd.read_excel()`. Luego enriquecemos la tabla principal **Ventas** uniendo las dimensiones mediante `merge()`.

In [ ]:
# ── Carga de todas las hojas ──────────────────────────────────────
FILE = 'ModeloComercial.xlsx'  # ajustar ruta si es necesario

hojas = pd.read_excel(FILE, sheet_name=None)

ventas    = hojas['Ventas'].copy()
paises    = hojas['País']
clientes  = hojas['Cliente']
vendedores= hojas['Vendedor']
operadores= hojas['Operador']
marcas    = hojas['Marca']
tiendas   = hojas['Tienda']
modelos   = hojas['Modelo']

print(f'Hojas cargadas: {list(hojas.keys())}')
print(f'\n📊 Tabla principal Ventas: {ventas.shape[0]:,} filas × {ventas.shape[1]} columnas')

In [ ]:
# ── Enriquecer tabla Ventas con dimensiones ───────────────────────
df = (ventas
      .merge(tiendas,    on='CódigoTienda',    how='left')
      .merge(vendedores, on='CódigoVendedor',  how='left')
      .merge(clientes,   on='CódigoCliente',   how='left')
      .merge(paises,     on='CódigoPaís',      how='left')
      .merge(operadores, on='CódigoOperador',  how='left')
      .merge(modelos[['CódigoModelo','DescripciónModelo','CódigoMarca']], on='CódigoModelo', how='left')
      .merge(marcas,     on='CódigoMarca',     how='left')
     )

# Convertir fechas
for col in ['Fecha','FechaVencimiento','FechaPago']:
    df[col] = pd.to_datetime(df[col])

# Variables derivadas
df['Año']          = df['Fecha'].dt.year
df['Mes']          = df['Fecha'].dt.month
df['Ganancia']     = df['Venta'] - (df['Costo'] * df['Cantidad'])
df['Margen_pct']   = (df['Ganancia'] / df['Venta'] * 100).round(2)
df['DiasRetraso']  = (df['FechaPago'] - df['FechaVencimiento']).dt.days

print(f'✅ Dataset enriquecido: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head(3)

---
## 🔍 2. Exploración Inicial

Usamos `info()`, `shape`, `head()`, `tail()` para entender la estructura del dataset y los tipos de datos de cada columna.

In [ ]:
print(f'📐 Dimensiones del dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'\n📅 Rango temporal: {df["Fecha"].min().date()} → {df["Fecha"].max().date()}')
print(f'\n📋 Columnas disponibles:')
for i, col in enumerate(df.columns, 1):
    print(f'   {i:02d}. {col}')

In [ ]:
# Tipos de datos y valores no nulos
df.info()

In [ ]:
# Primeras 5 filas
print('🔝 Primeras 5 filas:')
df.head(5)

In [ ]:
# Últimas 5 filas
print('🔚 Últimas 5 filas:')
df.tail(5)

---
## 🧹 3. Análisis de Calidad de Datos

Revisamos valores nulos, duplicados e inconsistencias que puedan afectar el análisis.

In [ ]:
# ── Valores nulos ─────────────────────────────────────────────────
nulos = pd.DataFrame({
    'Nulos':      df.isnull().sum(),
    'Porcentaje': (df.isnull().sum() / len(df) * 100).round(2)
})
nulos = nulos[nulos['Nulos'] > 0]

if nulos.empty:
    print('✅ No se encontraron valores nulos en el dataset.')
else:
    print('⚠️ Variables con valores nulos:')
    print(nulos)

In [ ]:
# ── Duplicados ────────────────────────────────────────────────────
dupl = df.duplicated().sum()
dupl_factura = df['Factura'].duplicated().sum()

print(f'Filas completamente duplicadas : {dupl}')
print(f'Facturas duplicadas            : {dupl_factura}')
print(f'\n✅ Facturas únicas             : {df["Factura"].nunique():,}')

# Verificar coherencia de precios
inconsistentes = df[df['Precio'] < df['Costo']]
print(f'\nVentas con Precio < Costo (inconsistencias): {len(inconsistentes)}')

In [ ]:
# ── Resumen de variables categóricas clave ────────────────────────
categoricas = ['RazónSocialTienda', 'NombreVendedor', 'RazónSocial',
               'DescripciónMarca', 'RazónSocialOperador']

for col in categoricas:
    print(f'\n{col}: {df[col].nunique()} categorías únicas')
    print('  ', df[col].unique().tolist())

---
## 📊 4. Estadística Descriptiva

Analizamos la distribución de las variables numéricas y el resumen de las categóricas para entender el comportamiento general del negocio.

In [ ]:
# ── Variables numéricas ───────────────────────────────────────────
print('📈 Estadística descriptiva — Variables numéricas:')
df[['Cantidad','Costo','Precio','Venta','Ganancia','Margen_pct','DiasRetraso']].describe().round(2)

In [ ]:
# ── Variables categóricas ─────────────────────────────────────────
print('📋 Estadística descriptiva — Variables categóricas:')
df[['RazónSocialTienda','NombreVendedor','DescripciónMarca',
    'RazónSocial','RazónSocialOperador']].describe(include='object')

In [ ]:
# ── KPIs principales del negocio ──────────────────────────────────
total_ventas   = df['Venta'].sum()
total_ganancia = df['Ganancia'].sum()
margen_global  = total_ganancia / total_ventas * 100
total_unidades = df['Cantidad'].sum()
ticket_prom    = df.groupby('Factura')['Venta'].sum().mean()

print('=' * 45)
print('       KPIs GENERALES DEL NEGOCIO')
print('=' * 45)
print(f'  Total Ventas       : S/ {total_ventas:>12,.0f}')
print(f'  Ganancia Bruta     : S/ {total_ganancia:>12,.0f}')
print(f'  Margen Global      :    {margen_global:>11.2f}%')
print(f'  Unidades Vendidas  :    {total_unidades:>12,}')
print(f'  Ticket Promedio    : S/ {ticket_prom:>12,.0f}')
print(f'  Período            :    2014 → 2020')
print(f'  Facturas           :    {df["Factura"].nunique():>12,}')
print('=' * 45)

### 💡 Variables más importantes para el análisis

| Variable | Tipo | Importancia |
|---|---|---|
| **Venta** | Numérica | Métrica principal de rendimiento |
| **Ganancia** | Numérica (derivada) | Indica la rentabilidad real |
| **Margen_pct** | Numérica (derivada) | Eficiencia por transacción |
| **Fecha / Año** | Temporal | Permite análisis de tendencia |
| **RazónSocialTienda** | Categórica | Comparación entre puntos de venta |
| **DescripciónMarca** | Categórica | Portafolio de producto |
| **NombreVendedor** | Categórica | Desempeño individual |
| **DiasRetraso** | Numérica (derivada) | Riesgo de cobranza |

---
## 📈 5. Visualizaciones con Plotly

Generamos 6 visualizaciones interactivas que cubren categorías, tendencias, distribuciones, proporciones, dispersión y valores atípicos.

In [ ]:
# ── VIZ 1: Barras horizontales — Ventas por Tienda ────────────────
vt = df.groupby('RazónSocialTienda').agg(
    Ventas=('Venta','sum'),
    Ganancia=('Ganancia','sum')
).reset_index().sort_values('Ventas')

fig1 = px.bar(
    vt, x='Ventas', y='RazónSocialTienda',
    orientation='h',
    text=vt['Ventas'].apply(lambda x: f'S/ {x/1e6:.1f}M'),
    color='Ventas',
    color_continuous_scale='Teal',
    title='📊 Ventas Totales por Tienda (S/)',
    labels={'Ventas':'Ventas (S/)','RazónSocialTienda':'Tienda'}
)
fig1.update_traces(textposition='outside')
fig1.update_layout(height=400, showlegend=False,
                   plot_bgcolor='#F8F9FA', paper_bgcolor='white',
                   coloraxis_showscale=False)
fig1.show()
print('\n💬 Interpretación: Tottus y Oeschle lideran con S/10.9M y S/10.3M respectivamente,'
      ' concentrando el 55% de las ventas totales. Sagabella registra el menor volumen (S/1.7M).')

In [ ]:
# ── VIZ 2: Línea — Evolución anual de ventas ─────────────────────
va = df.groupby('Año')['Venta'].sum().reset_index()

fig2 = px.line(
    va, x='Año', y='Venta',
    markers=True,
    text=va['Venta'].apply(lambda x: f'S/{x/1e6:.1f}M'),
    title='📈 Evolución de Ventas Anuales (2014–2020)',
    labels={'Venta':'Ventas (S/)','Año':'Año'}
)
fig2.update_traces(textposition='top center', line_color='#0D9488',
                   marker=dict(size=10, color='#0D9488'))
fig2.update_layout(height=400, plot_bgcolor='#F8F9FA', paper_bgcolor='white')
fig2.show()
print('\n💬 Interpretación: Las ventas alcanzaron su pico en 2017 (S/6.85M) y muestran una'
      ' tendencia decreciente hasta 2020 (S/3.2M), una caída del 53% en 3 años.')

In [ ]:
# ── VIZ 3: Dona — Participación por Marca ────────────────────────
vm = df.groupby('DescripciónMarca')['Venta'].sum().reset_index()

fig3 = px.pie(
    vm, values='Venta', names='DescripciónMarca',
    hole=0.45,
    title='🍩 Participación de Ventas por Marca (%)',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig3.update_traces(textinfo='label+percent', pull=[0.05,0,0,0,0,0])
fig3.update_layout(height=420, paper_bgcolor='white')
fig3.show()
print('\n💬 Interpretación: Samsung lidera con ~21.5% seguido de LG (20.4%) y Huawei (15.9%).'
      ' El mercado está bien diversificado entre 6 marcas sin dominancia extrema.')

In [ ]:
# ── VIZ 4: Boxplot — Distribución de Margen por Tienda ───────────
fig4 = px.box(
    df, x='RazónSocialTienda', y='Margen_pct',
    color='RazónSocialTienda',
    title='📦 Distribución de Margen % por Tienda (Boxplot)',
    labels={'Margen_pct':'Margen (%)','RazónSocialTienda':'Tienda'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig4.add_hline(y=df['Margen_pct'].mean(), line_dash='dash',
               line_color='red', annotation_text='Promedio global')
fig4.update_layout(height=420, showlegend=False,
                   plot_bgcolor='#F8F9FA', paper_bgcolor='white')
fig4.show()
print('\n💬 Interpretación: Sagabella presenta la mayor dispersión en margen y algunos'
      ' valores atípicos superiores. La mayoría de tiendas opera entre 4% y 8% de margen.')

In [ ]:
# ── VIZ 5: Histograma — Distribución del Monto de Venta ──────────
fig5 = px.histogram(
    df, x='Venta', nbins=40,
    title='📊 Distribución del Monto de Venta por Factura',
    labels={'Venta':'Monto de Venta (S/)','count':'Frecuencia'},
    color_discrete_sequence=['#0D9488']
)
fig5.add_vline(x=df['Venta'].mean(), line_dash='dash',
               line_color='red', annotation_text=f'Media: S/{df["Venta"].mean():,.0f}')
fig5.add_vline(x=df['Venta'].median(), line_dash='dot',
               line_color='orange', annotation_text=f'Mediana: S/{df["Venta"].median():,.0f}')
fig5.update_layout(height=400, plot_bgcolor='#F8F9FA', paper_bgcolor='white')
fig5.show()
print('\n💬 Interpretación: La distribución es asimétrica hacia la derecha. La mayoría'
      ' de facturas se concentra entre S/2,000 y S/10,000, con algunas ventas de alto'
      ' valor que elevan la media por encima de la mediana.')

In [ ]:
# ── VIZ 6: Scatter — Venta vs Ganancia por Tienda ────────────────
fig6 = px.scatter(
    df, x='Venta', y='Ganancia',
    color='RazónSocialTienda',
    size='Cantidad',
    hover_data=['NombreVendedor','DescripciónMarca','Año'],
    title='🔵 Relación Venta vs Ganancia por Tienda',
    labels={'Venta':'Venta (S/)','Ganancia':'Ganancia (S/)'},
    color_discrete_sequence=px.colors.qualitative.Set1,
    opacity=0.6
)
fig6.update_layout(height=450, plot_bgcolor='#F8F9FA', paper_bgcolor='white')
fig6.show()
print('\n💬 Interpretación: Existe correlación positiva entre venta y ganancia, pero la'
      ' pendiente varía entre tiendas. Algunos puntos con alta venta tienen ganancia'
      ' relativamente baja, evidenciando diferencias en la eficiencia comercial.')

---
## 🦆 6. Consultas SQL con DuckDB

Aplicamos DuckDB para ejecutar consultas SQL directamente sobre el DataFrame `df` de pandas, sin necesidad de base de datos externa. Esto nos permite combinar la flexibilidad de Python con la expresividad de SQL.

**DuckDB registra el DataFrame automáticamente cuando se referencia por nombre en la query.**

In [ ]:
# Conexión DuckDB
con = duckdb.connect()
print('✅ Conexión DuckDB establecida')
print(f'   El DataFrame "df" tiene {len(df):,} filas listas para consultar')

In [ ]:
# ── SQL 1: SELECT + LIMIT — Inspección de las primeras filas ──────
print('📋 SQL 1 — Inspección inicial con SELECT y LIMIT')
print('-' * 55)

q1 = con.execute("""
    SELECT
        Factura,
        Fecha,
        RazónSocialTienda AS Tienda,
        NombreVendedor    AS Vendedor,
        DescripciónMarca  AS Marca,
        Cantidad,
        Venta,
        Ganancia,
        ROUND(Margen_pct, 2) AS Margen_pct
    FROM df
    ORDER BY Fecha DESC
    LIMIT 10
""").df()

q1

In [ ]:
# ── SQL 2: WHERE — Ventas con margen superior al promedio ─────────
print('🔍 SQL 2 — Filtrado con WHERE: facturas con margen > promedio global (5.55%)')
print('-' * 65)

q2 = con.execute("""
    SELECT
        Factura,
        RazónSocialTienda  AS Tienda,
        NombreVendedor     AS Vendedor,
        DescripciónMarca   AS Marca,
        Venta,
        Ganancia,
        ROUND(Margen_pct, 2) AS Margen_pct
    FROM df
    WHERE Margen_pct > 5.55
    ORDER BY Margen_pct DESC
    LIMIT 10
""").df()

total_sobre_media = con.execute("SELECT COUNT(*) FROM df WHERE Margen_pct > 5.55").fetchone()[0]
print(f'   Facturas con margen > 5.55%: {total_sobre_media:,} de {len(df):,} ({total_sobre_media/len(df)*100:.1f}%)')
q2

In [ ]:
# ── SQL 3: GROUP BY — Ventas y margen promedio por tienda ─────────
print('📊 SQL 3 — Agregación con GROUP BY: rendimiento por tienda')
print('-' * 60)

q3 = con.execute("""
    SELECT
        RazónSocialTienda                   AS Tienda,
        COUNT(DISTINCT Factura)             AS Facturas,
        SUM(Cantidad)                       AS Unidades,
        ROUND(SUM(Venta) / 1e6, 2)         AS Ventas_M,
        ROUND(SUM(Ganancia) / 1e3, 1)      AS Ganancia_K,
        ROUND(AVG(Margen_pct), 2)          AS Margen_Prom_pct,
        ROUND(SUM(Venta)*100.0 /
              SUM(SUM(Venta)) OVER(), 1)   AS Part_pct
    FROM df
    GROUP BY RazónSocialTienda
    ORDER BY Ventas_M DESC
""").df()

q3

In [ ]:
# ── SQL 4: ORDER BY — Ranking de vendedores por ganancia ──────────
print('🏆 SQL 4 — Ranking con ORDER BY: top vendedores por ganancia generada')
print('-' * 60)

q4 = con.execute("""
    SELECT
        ROW_NUMBER() OVER (ORDER BY SUM(Ganancia) DESC) AS Ranking,
        NombreVendedor                                   AS Vendedor,
        COUNT(DISTINCT Factura)                          AS Facturas,
        SUM(Cantidad)                                    AS Unidades,
        ROUND(SUM(Venta) / 1e6, 2)                      AS Ventas_M,
        ROUND(SUM(Ganancia) / 1e3, 1)                   AS Ganancia_K,
        ROUND(AVG(Margen_pct), 2)                        AS Margen_pct
    FROM df
    GROUP BY NombreVendedor
    ORDER BY Ganancia_K DESC
""").df()

q4

In [ ]:
# ── SQL 5: Consulta compleja — Clientes con pago tardío ───────────
print('⏱️ SQL 5 — Consulta avanzada: clientes con retraso en pago + impacto financiero')
print('-' * 70)

q5 = con.execute("""
    SELECT
        RazónSocial                             AS Cliente,
        COUNT(DISTINCT Factura)                 AS Facturas,
        ROUND(SUM(Venta) / 1e6, 2)             AS Ventas_M,
        ROUND(AVG(DiasRetraso), 1)              AS Dias_Retraso_Prom,
        SUM(CASE WHEN DiasRetraso > 0
                 THEN 1 ELSE 0 END)             AS Pagos_Tardios,
        ROUND(SUM(CASE WHEN DiasRetraso > 0
                  THEN 1.0 ELSE 0 END)
              / COUNT(*) * 100, 1)              AS Pct_Tardio,
        CASE
            WHEN AVG(DiasRetraso) > 0 THEN '🔴 Riesgo'
            WHEN AVG(DiasRetraso) = 0 THEN '🟡 Puntual'
            ELSE '🟢 Anticipado'
        END                                     AS Estado_Cobranza
    FROM df
    GROUP BY RazónSocial
    ORDER BY Dias_Retraso_Prom DESC
""").df()

q5

In [ ]:
# ── SQL 6 (BONUS): Tendencia anual con crecimiento YoY ────────────
print('📅 SQL 6 — Tendencia anual con variación año a año (YoY)')
print('-' * 60)

q6 = con.execute("""
    WITH ventas_anuales AS (
        SELECT
            Año,
            SUM(Venta)    AS Ventas,
            SUM(Ganancia) AS Ganancia,
            COUNT(DISTINCT Factura) AS Facturas
        FROM df
        GROUP BY Año
    )
    SELECT
        Año,
        ROUND(Ventas / 1e6, 2)                                AS Ventas_M,
        ROUND(Ganancia / 1e3, 1)                              AS Ganancia_K,
        Facturas,
        ROUND(
            (Ventas - LAG(Ventas) OVER (ORDER BY Año))
            / LAG(Ventas) OVER (ORDER BY Año) * 100, 1
        )                                                     AS Var_YoY_pct
    FROM ventas_anuales
    ORDER BY Año
""").df()

q6

---
## 🏁 7. Hallazgos y Conclusiones

### 📌 Principales hallazgos del análisis

| # | Hallazgo | Detalle |
|---|---|---|
| 1 | **Concentración de ventas** | Tottus y Oeschle representan el 55% de las ventas totales (S/21.2M de S/38.7M) |
| 2 | **Brecha volumen-margen** | Oeschle vende casi igual que Tottus, pero con un margen ~0.5 p.p. inferior — gran oportunidad de mejora |
| 3 | **Tendencia decreciente** | Las ventas caen un 53% entre 2017 y 2020, señal de alerta estratégica |
| 4 | **Liderazgo Samsung** | Samsung lidera ventas por marca (~21.5%), seguido de LG y Huawei |
| 5 | **Desempeño desigual** | María genera S/13.4M (35% del total) vs Luisa con S/4.2M — alta dependencia de un vendedor |
| 6 | **Cobranza saludable** | Solo Corporación Moderna y 2 clientes más pagan con retraso; la mayoría paga puntual o anticipado |
| 7 | **Margen global ajustado** | El margen bruto global es 5.55% — bajo para la industria, lo que sugiere revisar precios |

### 🎯 Recomendaciones

1. **Revisar la estrategia de precios en Oeschle** — mejorar el mix hacia modelos Samsung/Apple de mayor margen.
2. **Investigar la caída 2017–2020** — puede ser pérdida de mercado, cambio de canal o efecto macroeconómico.
3. **Diversificar la fuerza de ventas** — reducir la dependencia de María implementando planes de desarrollo para el equipo.
4. **Fortalecer el seguimiento a Corporación Moderna** — único cliente con promedio de retraso positivo.

---

> *Análisis desarrollado con Python · pandas · DuckDB · Plotly*  
> *Dataset: ModeloComercial.xlsx — 1,499 facturas · 7 tiendas · 6 marcas · 2014–2020*